# Day 2 · 버그 사냥 — "돌아간다"와 "맞다" 사이
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kehdgus9434/sonic-vibe-coding-lab/blob/main/notebooks/day2_bughunt.ipynb)

**오늘의 결과물**: 버그 4개를 고친 폐루프 파이프라인 + 참조 구현 없이도 버그를 잡는 불변식 테스트 + 40초 완주 영상

| 시간 | 내용 |
|---|---|
| 0:00–0:10 | 환경 준비 · 증상 관찰 |
| 0:10–1:05 | Mission 2-1 버그 사냥 (4개) |
| 1:05–1:30 | Mission 2-2 불변식 테스트 만들기 |
| 1:30–1:50 | Mission 2-3 40초 완주 + 영상 |
| 1:50–2:00 | 회고 |

In [ ]:
#@title 0. 실습 환경 준비 — 처음 한 번 실행 (약 1–2분)
REPO = "https://github.com/kehdgus9434/sonic-vibe-coding-lab"   # 에이전트형 도구 사용자는 내 Fork 주소로 바꾸세요
import os, sys, subprocess
from pathlib import Path
if Path("/content").exists():                              # Google Colab
    LAB_DIR = Path("/content/sonic-vibe-coding-lab")
    if not LAB_DIR.exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO, str(LAB_DIR)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(LAB_DIR / "requirements-colab.txt")], check=True)
else:                                                      # 로컬 Jupyter (저장소 안에서 실행)
    LAB_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(LAB_DIR)
sys.path.insert(0, str(LAB_DIR))
import numpy as np
import sonic_lab
from sonic_lab import check, reference as ref, render, analysis
lab = sonic_lab.start()

In [ ]:
# 버그 사냥 대상 불러오기 — 파일을 고칠 때마다 이 셀부터 다시 실행
import importlib.util
def load_buggy(path="exercises/buggy_pipeline.py"):
    spec = importlib.util.spec_from_file_location("buggy_pipeline", path)
    module = importlib.util.module_from_spec(spec); spec.loader.exec_module(module)
    return module.Pipeline()
pipeline = load_buggy()

## 증상 관찰 (5분)
AI가 작성한 파이프라인(`exercises/buggy_pipeline.py`)은 오류 없이 실행됩니다. 결과를 보세요.

In [ ]:
bug_summary, bug_arrays = ref.run_physics(lab.sim, lab.policy, lab.motion, mode=2, max_frames=250, pipeline=pipeline)
print({k: bug_summary[k] for k in ["simulated_frames", "requested_frames", "stop_reason"]})
render.show(render.skeleton_video(lab, bug_arrays, "outputs/day2_buggy.mp4", title="buggy pipeline", seconds=5))

### 📝 프롬프트 로그 (이 셀을 더블클릭해 편집)
| # | 내가 한 요청 | AI 답 요약 | 내가 확인한 근거 | 채택? |
|---|---|---|---|---|
| 1 | | | | |
| 2 | | | | |
| 3 | | | | |

---
## Mission 2-1 · 버그 사냥 (55분)

- **대상**: `exercises/buggy_pipeline.py` (Colab 왼쪽 📁 → `sonic-vibe-coding-lab/exercises/`에서 더블클릭해 편집)
- **버그**: 계약서와 다른 곳이 **4개**. 세 함수(`observation`, `decoder_obs`, `action_to_target`)에 흩어져 있고, 한 함수에 2개가 있을 수 있습니다.
- **진행**
  1. 파일 전체 + `docs/01_contract.md` + 아래 ✅ 출력을 AI에게 주고 **원인 후보와 그것을 가려낼 테스트**를 먼저 요청합니다(`docs/02_vibe_coding_guide.md`의 디버깅 템플릿).
  2. 테스트로 확인된 것만 고칩니다. 고칠 때마다 파일 맨 위 `BUGLOG`에 "무엇을, 어떤 근거로"를 적습니다.
  3. 파일 저장 → `load_buggy` 셀 → ✅ 셀 재실행.
- **완료 기준**: 구성요소 3/3 통과 + 폐루프 250프레임 완주
- ⚠️ Colab 런타임이 끊기면 파일 수정도 사라집니다. 고친 코드를 수시로 아래 메모 셀에 복사해 두세요.

In [ ]:
# ✅ 확인 (고칠 때마다 load_buggy 셀 → 이 셀)
pipeline = load_buggy()
_ = check.bughunt(lab, pipeline, frames=250)

```python
# 📋 고친 코드 백업용 메모 (런타임이 끊기면 여기서 복구)
```

---
## Mission 2-2 · 불변식 테스트 만들기 (25분)
> 🧭 **방법**: 아래 설명 + `docs/01_contract.md` + 함수 시그니처를 바이브 코딩 도구에 주고 ✍️ 셀을 채운 뒤, ✅ 셀로 확인합니다.

방금은 정답(참조 구현)과 비교하는 `check`가 있어서 버그를 찾을 수 있었습니다. **실제 연구에서는 정답이 없습니다.**
참조 구현 없이도 같은 버그를 잡을 수 있는 테스트를 AI와 함께 만드세요.

- **입력**: `pipeline`(검사 대상), `lab`(`lab.sim`, `lab.contract` 사용 가능 · `ref.*` 정답 함수 결과와 비교 금지)
- **방법**: 물리적으로 당연한 성질을 `assert`로. 예를 들어 이런 질문에서 시작하세요.
  - 로봇이 기본 자세로 서 있으면 관측 `q`는 어떤 값이어야 할까?
  - 기저가 똑바로 서 있으면 `gravity`는? 기저를 z축으로 90° 돌리면?
  - 이력에 프레임 번호를 값으로 넣으면 Decoder 입력 안에서 어떤 순서로 나타나야 할까?
  - `action = 0`이면 목표 관절각은? 한 관절만 1이면 **어떤 이름의** 관절이 움직여야 할까?
- **완료 기준**: 올바른 파이프라인은 통과, 버그 4유형을 각각 따로 넣은 변형은 모두 `AssertionError`

In [ ]:
# ✍️ Mission 2-2
import mujoco
from scipy.spatial.transform import Rotation as R

def invariant_tests(pipeline, lab):
    """참조 구현 없이 버그를 잡는 테스트. 문제가 있으면 AssertionError."""
    sim, c = lab.sim, lab.contract
    raise NotImplementedError

# from my_work.day2 import *

In [ ]:
# ✅ 확인 — 버그 변형 4종에 대해 내 테스트를 돌립니다
check.invariant_tests(lab, invariant_tests)

---
## Mission 2-3 · 40초 완주와 영상 (20분)
고친 파이프라인으로 공식 샘플 전체(2,002프레임)를 실행하고, 참조 구현과 수치를 비교합니다.
같아야 할까요, 달라도 될까요? 다르다면 얼마나까지 괜찮을까요?

In [ ]:
pipeline = load_buggy()
fixed = ref.run_physics(lab.sim, lab.policy, lab.motion, mode=2, pipeline=pipeline, verbose=True)
reference = ref.run_physics(lab.sim, lab.policy, lab.motion, mode=2, verbose=True)
analysis.table({"내 파이프라인": fixed, "참조 구현": reference})

In [ ]:
# 영상 (GPU 런타임이면 EGL, CPU면 OSMesa 소프트웨어 렌더링 · 6초)
video = render.mesh_video(lab, fixed[1], "outputs/day2_fixed.mp4", label="fixed pipeline / mode 2", seconds=6)
render.show(video or render.skeleton_video(lab, fixed[1], "outputs/day2_fixed_skeleton.mp4", title="fixed", seconds=6))

---
## 회고 (10분)
- 4개 버그 중 AI가 **먼저** 찾은 것과 **내가 먼저** 의심한 것은?
- "에러 없이 돌아가는 코드"가 왜 더 위험했나요?
- Mission 2-2의 테스트 중 실제 연구 코드에 그대로 남겨 두고 싶은 것은?
- **Day 3 예고**: 직접 실험을 설계하고 3분 발표합니다. 어떤 질문을 해 볼지 생각해 오세요(과제 아님).